# ISS_preprocessing of Leica TIFF and LIF files

This notebook guides you through the preprocessing of TIFF files, as autosaved or exported by Leica microscopes, and LIF files, with associated Metadata.
 

To use this notebook the you must have:
1) **Individual TIFF files**, autosaved or exported, directly from the Leica software. Each tiff file will represent a single plane of a single tile of a single channel, so thousands of individual TIFFs will be created in a typical experiment.
2) **A single LIF file** (containing single or multiple regions) **or multiple LIF files** (one for each region), saved directly from the Leica software.

For our script to work it is important that the TIFF files are rigorously and consistently indexed, and this guaranteed by the following example naming structure:

### For autosaved TIFF files:

`TileScan 1--Stage99--Z42--C03.tif`, where 

- `TileScan` represents the ROI, 

- `Stage` represents the tile, 

- `Z` represent Z-plane,

- `C` represents the channel, and  

- the `--`  sign acts as a separator (default on our Leica microscopes).

### For exported TIFF files:

`2024-10-24--12-31-35-538 TEST_sample21_cycle1_2_s00_z42_ch3.tif`, where 

- `s` represents the tile, 

- `z` represent Z-plane,

- `ch` represents the channel, and  

- the `_`  sign acts as a separator (default on our Leica microscopes).

There are no contstrains on the naming for LIF files, but **it is important that files for EACH CYCLE are stored in SEPARATE FOLDERS,** so that the file path would resemble `'/path/to/cycle1/your_lif_file.lif'` etc.


We begin importing the necessary libraries and tools

In [1]:
import ISS_preprocessing.preprocessing as pp


## This notebook offers deconvolution using `RedLionFish/Deconwolf`

`RedLionFish` is a GPU-accelerated deconvolution tool and to run it **you need a CUDA-compatible GPU with functional NVIDIA drivers, cuDNN, etc...**
`Deconwolf` is a CPU-based deconvolution tool. The Deconwolf executable is a command-line tool and **needs to be installed before using, see deconwolf_install.txt.** You can skip this installation if using only `RedLionFish`.

To do image deconvolution, you will first need to know some parameters about your microscope. Take some time to carefully read the guide below and to collect some information before you attempt image deconvolution. This will save you a lot of computing time and frustration: using the wrong parameters will result in imaging artifacts, so be careful!

These parameters will be used to create a synthetic point-spread function (PSF). If you don't know what a PSF is and why it is important, please read here: ​​https://en.wikipedia.org/wiki/Point_spread_function

You will need to know:

`na` = numerical aperture of the used len

`m` = lens magnification

`ni0` = refraction index of the immersion medium

`res_lateral` = x,y resolution of the images 

`res_axial`: z-resolution of the stack. This is either the spacing between zplanes or the actual z resolution of your lens, whichever is larger. Remember that the actual Z resolution of your lens will match your experiment’s resolution only if you acquired the stacks at Nyquist conditions. (https://imb.uq.edu.au/research/facilities/microscopy/training-manuals/microscopy-online-resources/image-capture/nyquist-conditions)


## Introduce the PSF_metadata

Once you've collected the above information, you have to input it in the format indicated below. You can substitute the numbers with the actual values for your microscope. **The PSF is ONLY needed when doing deconvolution.**


In [2]:
#THIS IS FOR 5 COLOURS LEICA 20X
PSF_metadata = {'na':1.1,
'm':20,
'ni0':1.333,
'res_lateral':0.419,
'res_axial':0.859,
 'channels':{
 '0':{
    'wavelength':.809},
  '1':{
    'wavelength':.681},
  '2':{
    'wavelength':.555},
  '3':{
    'wavelength':.475},
  '4':{
    'wavelength':.390},
  '5':{
    'wavelength':.436}
     
 }
}

## Main function for Leica preprocessing
This function processes Leica-exported TIFF and LIF files and under the hood runs many functions that are made invisible for convenience. It takes as input 3D imaging stacks, performs deconvolution (optional), projects the 3D stacks to 2D images, aligns and stitches the images, and finally outputs retiled images with desired dimensions. Once if you figured out the right parameters for your specific microscope, this will be the smoother way of running preprocessing.

**The function itself processes one cycle at a time.** This means that the specified input directory contains files for ONE CYCLE and you need to manually specify which cycle the input directory refers to for appropriate naming of the output files. **However, in this notebook, the function is embedded in a loop to process multiple cycles.**

**This function is able to handle multiple regions** in the input files, and project them accordingly.

In the function you will have to define the following variables:

`input_dir`: This will be the complete path to the folder containing your one imaging cycle.  The format for this variable is a `str`.

`output_dir_prefix` = This will be the path where you want the preprocessing output to be saved. Ideally, this should be associated with some type of unique project identifier. The format of this variable is `str`. Subfolders for each one of the scanned regions will be created as `R1`, `R2`, etc...

`cycle`: This will be an `int` number of the ISS cycle the input_dir refers to, where 1 refers to cycle 1 and so on. If `cycle=None` the function will not work.

`mode`: `mode='tif_autosaved'` for autosaved TIFF files in Leica microscope, `mode='tif_exported'` for exported TIFF files in Leica microscope and `mode='lif'` for LIF files. 

`deconvolution_method`: choose between `deconvolution_method='redlionfish'` (gpu based) and `deconvolution_method='deconwolf'` (cpu based). The Deconwolf executable is a command-line tool and needs to be installed before using, see deconwolf_install.txt. Choose `deconvolution_method=None` to skip deconvolution as a first step in the preprocessing.

`PSF_metadata`: refer to the example above. Metadata for the construction of a synthetic PSF. Default is `None`. This can be skipped if `deconvolution_method=None`.

`mip`: Specifies if the deconvolved images need to be maximum projected (default = True). If `False` the deconvolved stacks are saved, however we do our ISS analysis in 2D so there's almost never a good reason to save the stack.

`align_channel`: This variable sets on which channel the alignment across cycles is performed, and typically points to the DAPI channel. This is an `int` number, which referes to the channel number containing the DAPI images. The number is relative to the order of channel acquisition on your  microscope. In our Leica set up, this is the fifth channel, which in python would mean that we put 4 (since python is zero indexed). Default is 4.

`tile_dimension` =  This `int` refers to the number of pixels that you want to tile your images into during the reslicing process. Default is 6000, i.e. your resliced images will be of the shape 6000x6000 pixels. 





### Time to start processing!
Now that you have read **ALL the instructions above**, you are ready to process your files. Below is the `preprocessing_main_leica` embeded in **a for loop to process MULTIPLE CYCLES**. 

User specifies:

`input_dirs`: a list of input directories for each cycle. When processing **one cycle** only, `input_dirs` is a list of one input directory.

`cycles`: a list of cycles corresponding to the input directories. When processing **one cycle** only,  `cycles` is a list of one cycle corresponding to the input directory.


In [ ]:
input_dirs = ['/path/to/cycle1/',
              '/path/to/cycle2/',
              '/path/to/cycle3/',
              '/path/to/cycle4/',
              '/path/to/cycle5/']

output_dir_prefix='/path/to/my/output/folder/'

cycles = [1,2,3,4,5]

for cycle, input_dir in zip(cycles, input_dirs):

    pp.preprocessing_main_leica(
                    input_dir,
                    output_dir_prefix='/path/to/my/output/folder/',
                    cycle=cycle,
                    mode='tif_autosaved',
                    deconvolution_method='redlionfish',
                    PSF_metadata=PSF_metadata,
                    align_channel=4,
                    tile_dimension=6000)

## Access to individual functions for Leica processing

Instead of running the main function as outlined above, we can also choose to run the step by step subfunctions one at a time. 

**These functions processes only ONE CYCLE at a time.** This means that you need to manually specify which cycle the file refers to for appropriate naming of the output files. However, they do process **multiple regions** in one go.

Let's have a quick look at what each function does.


### `deconvolve_leica`

`deconvolve_leica` is the function to **deconvolve and maximum-project the images from the input folders**. It takes the following arguments, which mirror the same arguments of the main function:

`input_dir`: This will be the complete path to the folder containing your **ONE imaging cycle**.  The format for this variable is a `str`.

`output_dir_prefix` = This will be the path where you want the preprocessing output to be saved. Ideally, this should be associated with some type of unique project identifier. The format of this variable is `str`. Subfolders for each one of the scanned regions will be created as `_R1`, `_R2`, etc...

`cycle`: This will be an `int` number of the ISS cycle the input_dir refers to, where 1 refers to cycle 1 and so on. If `cycle=None` the function will not work.

`mode`: `mode='tif_autosaved'` for autosaved TIFF files in Leica microscope, `mode='tif_exported'` for exported TIFF files in Leica microscope and `mode='lif'` for LIF files. 

`deconvolution_method`: choose between `deconvolution_method='redlionfish'` (gpu based) and `deconvolution_method='deconwolf'` (cpu based). The Deconwolf executable is a command-line tool and needs to be installed before using, see deconwolf_install.txt. Choose `deconvolution_method=None` to skip deconvolution as a first step in the preprocessing.

`PSF_metadata`: refer to the example above. Metadata for the construction of a synthetic PSF. Default is `None`. This can be skipped if `deconvolution_method=None`.

`mip`: Specifies if the deconvolved images need to be maximum projected (default = True). If `False` the deconvolved stacks are saved, however we do our ISS analysis in 2D so there's almost never a good reason to save the stack.


The function outputs deconvolved and mipped images per cycle into the `/preprocessing/Cycle{cycle}/1_mipped/` subfolder, **as well as a list of region directories that can be used in the following functions below**.



In [ ]:
input_dir = '/path/to/cycle1/'
output_dir_prefix = '/path/to/my/output/folder/'
cycle = 1

In [ ]:
region_directories = pp.deconvolve_leica(
                        input_dir,
                        output_dir_prefix, 
                        cycle,
                        mode='tif_exported',
                        deconvolution_method='redlionfish',
                        PSF_metadata=PSF_metadata
                        )

### `mipped_to_OME_tiffs`
`mipped_to_OME_tiffs` is the function that **takes the projected images across channels and wraps them into a single OMEtiff per imaging cycle**. This steps organises the files corresponding to each imaging cycles in a specific way within a single file and requires the parsing of a Metadata file to arrange correctly the images in xy space. 

It takes the following arguments, which mirror the same arguments of the main function:

`region_directories`: list of directories for each region one wishes to process. The output from `deconvolve_leica` can be used here.
 
`cycle`: This will be an `int` number of the ISS cycle the input_dir refers to, where 1 refers to cycle 1 and so on. If `cycle=None` the function will not work.

Mipped images need to be accessible in `/preprocessing/Cycle{cycle}/1_mipped/` subfolder in each region directory.

The function outputs one OMEtiff file per cycle into the `/preprocessing/Cycle{cycle}/2_ome_tiffs/` subfolder.



In [ ]:
pp.mipped_to_OME_tiffs(
    region_directories,
    cycle
    )

### `align_and_stitch`

This function runs `ashlar`, a package for image stitching and cycle alignment. The function uses the OME_tiffs files as an input, takes as input a channel number (normally the DAPI, see above) and on that channels performs all the alignment and stitching operations.

It takes the following arguments, which mirror the same arguments of the main function:

`region_directories`: list of directories for each region one wishes to process.
 
`cycle`: This will be an `int` number of the ISS cycle the input_dir refers to, where 1 refers to cycle 1 and so on. If `cycle=None` the function will not work.

`align_channel`: This variable sets on which channel the alignment across cycles is performed, and typically points to the DAPI channel. This is an `int` number, which referes to the channel number containing the DAPI images. The number is relative to the order of channel acquisition on your  microscope. In our Leica set up, this is the fifth channel, which in python would mean that we put 4 (since python is zero indexed).  

OMEtiff file needs to be accessible in `/preprocessing/Cycle{cycle}/2_ome_tiff/` subfolder in each region directory.

The function outputs one stitched file per cycle and channel into the `/preprocessing/Cycle{cycle}/3_stitched/` subfolder

In [ ]:
pp.align_and_stitch(
    region_directories,
    cycle,
    align_channel=4
)

### `retile_stitched_images`
In this function the stitched images are re-tiled according to a user-specified size.
The reason for this is that stitched images are too big to be decoded directly and we prefer to decode them in tiles. This has several advantages, most notably that the pipeline would work also on laptops or non-powerful computers. The idea tile size is 4000-6000, but larger or smaller are also fine depending on the computer.

It takes the following arguments, which mirror the same arguments of the main function:

`region_directories`: list of directories for each region one wishes to process.
 
`cycle`: This will be an `int` number of the ISS cycle the input_dir refers to, where 1 refers to cycle 1 and so on. If `cycle=None` the function will not work.

`tile_dimensions` =  This `int` refers to the number of pixels that you want to tile your images into during the reslicing process. Default is 6000, i.e. your resliced images will be of the shape 6000x6000 pixels. 


The stitched files need to be accessible in `/preprocessing/Cycle{cycle}/3_ome_tiffs/` subfolder in each region directory.

The function outputs one stitched file per cycle and channel into the `/preprocessing/Cycle{cycle}/4_retiled/` subfolder

In [ ]:
pp.retile_stitched_images(
    region_directories,
    cycle,
    tile_dimension=6000
)